# 1. Anomaly Detection with Isolation Forest

In [1]:
import pandas as pd

In [2]:
from pathlib import Path


def _find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pricepoint").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the project root (a directory containing 'pricepoint/') "
        "above the notebook's current working directory."
    )


PROJECT_ROOT = _find_project_root()
FINAL_DATA_PATH = str(PROJECT_ROOT / "data" / "02_processed" / "feature_engineered_data.parquet")
df = pd.read_parquet(FINAL_DATA_PATH)
df.head(3)

,supermarket,prices,prices_unit,unit,product_name,date,category,own_brand,normalised_name,canonical_name,...,price_diff_1d,price_vs_market_avg,price_rank,is_cheapest_in_market,day_of_week_sin,day_of_week_cos,day_of_month_sin,day_of_month_cos,week_of_year_sin,week_of_year_cos
0,Morrisons,1.45,2.9,kg,Morrisons 0% Fat Greek Style…,2024-01-09,fresh_food,True,0 fat greek style,0 fat greek style,...,NaN,NaN,NaN,0,0.781831,0.623490,0.968077,-0.250653,0.239316,0.970942
1,Morrisons,1.45,2.9,kg,Morrisons 0% Fat Greek Style…,2024-01-10,fresh_food,True,0 fat greek style,0 fat greek style,...,0.0,NaN,NaN,0,0.974928,-0.222521,0.897805,-0.440394,0.239316,0.970942
2,Morrisons,1.45,2.9,kg,Morrisons 0% Fat Greek Style…,2024-01-11,fresh_food,True,0 fat greek style,0 fat greek style,...,0.0,NaN,NaN,0,0.433884,-0.900969,0.790776,-0.612106,0.239316,0.970942


In [3]:
from sklearn.ensemble import IsolationForest

# Use a subset of features relevant for detecting unusual pricing
anomaly_features = [
    'prices',
    'price_diff_1d',
    'price_rol_std_7d',
    'price_vs_market_avg'
]

# Drop NaNs from the selected features
df_anomaly = df[anomaly_features].dropna()

iso_forest = IsolationForest(n_estimators=100, contamination=0.01, random_state=123)
iso_forest.fit(df_anomaly)

# Predict anomalies on the cleaned subset
df_anomaly['anomaly_score'] = iso_forest.predict(df_anomaly)
df['anomaly'] = iso_forest.predict(df[anomaly_features].fillna(0))

anomalies = df[df['anomaly'] == -1]
print(f"Detected {len(anomalies)} potential anomalies.")
print("\nExamples of detected price anomalies:")
print(anomalies[['supermarket', 'canonical_name', 'date', 'prices', 'price_lag_1d', 'price_diff_1d']].head(10))


Detected 58857 potential anomalies.

Examples of detected price anomalies:
       supermarket    canonical_name       date  prices  price_lag_1d  \
117616       Sains  255cm frying pan 2024-01-15    26.0          14.0   
117637       Sains  255cm frying pan 2024-01-19    26.0          14.0   
117646       Sains  255cm frying pan 2024-01-21    26.0          14.0   
117651       Sains  255cm frying pan 2024-01-22    26.0          14.0   
117664       Sains  255cm frying pan 2024-01-24    26.0           5.0   
117669       Sains  255cm frying pan 2024-01-25    26.0          14.0   
117690       Sains  255cm frying pan 2024-01-30    26.0           5.0   
117700       Sains  255cm frying pan 2024-02-01    26.0           5.0   
117728       Sains  255cm frying pan 2024-02-08    26.0          14.0   
117734       Sains  255cm frying pan 2024-02-09    26.0          14.0   

        price_diff_1d  
117616           12.0  
117637           12.0  
117646           12.0  
117651           12.0  
1

Detected Anomalies: We have identified ~95,000 potential anomalies, which is roughly 1% of the dataset. 

Interpreting the Example: "3 tier steamer" at ASDA
On 2024-01-09, the price is £23.00, but the previous day's price (price_lag_1d) was £12.00. This is a huge jump (price_diff_1d = £11.00). The model flags this as an anomaly.

On 2024-01-10, the price drops back to £12.00. This is another massive change (price_diff_1d = -£11.00). The model flags this too.
We see this pattern repeating: the price seems to be oscillating daily between two distinct price points (£23.00, £12.00, and later £8.00).

Meaning of pattern:

Promotional Pricing: This could be a "deal of the day" type of promotion that is being turned on and off.

A/B Price Testing: The retailer might be actively testing price sensitivity by showing different prices to different user groups or at different times.

Data Scraping Error: It's possible that the scraper is hitting two different versions of the product page that have different prices, leading to this daily flip-flop.